# Инференс на полном test split

В настройках ноутбука выберите Accelerator → GPU T4. Добавьте `model.py`, `inference.py`, `shared_tokenizer.json` и inference checkpoint как Kaggle Dataset или загрузите их в `/kaggle/working`. Ноутбук скачает официальный test split, прогонит все примеры и сохранит `predictions.jsonl`.

In [ ]:
from pathlib import Path
import sys

def find_file(name):
    paths = [Path('/kaggle/working') / name]
    paths += list(Path('/kaggle/input').rglob(name))
    path = next((path for path in paths if path.exists()), None)
    if path is None:
        raise FileNotFoundError(f'Добавьте {name} в Kaggle notebook')
    return path

shared_tokenizer_path = find_file('shared_tokenizer.json')
checkpoint_path = next(
    (path for name in ('inference-best.pt', 'inference.pt', 'best.pt', 'model.pt')
     for path in [Path('/kaggle/working') / name, *Path('/kaggle/input').rglob(name)]
     if path.exists()),
    None,
)
if checkpoint_path is None:
    raise FileNotFoundError('Добавьте inference checkpoint в Kaggle notebook')
project_dir = find_file('inference.py').parent
sys.path.insert(0, str(project_dir))
print('project_dir:', project_dir)
print('shared tokenizer:', shared_tokenizer_path)
print('checkpoint:', checkpoint_path)

In [ ]:
import json
import subprocess
import time

import torch
from datasets import load_dataset
from tokenizers import Tokenizer
from tqdm.auto import tqdm

from inference import generate_texts, load_model

print('torch:', torch.__version__)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'Включите GPU Accelerator в настройках Kaggle'


In [ ]:
TEST_FILE = 'hf://datasets/SlavaYus/tinysms_qa_12m/data/test/test.parquet'
BATCH_SIZE = 64
MAX_SOURCE_LENGTH = 192
MAX_NEW_TOKENS = 256
NUM_BEAMS = 3
LENGTH_PENALTY = 1.0
WARMUP_SIZE = 8

In [ ]:
test = load_dataset(
    'parquet',
    data_files={'test': TEST_FILE},
    split='test',
)
prompts = test['question']
source_tokenizer = Tokenizer.from_file(str(source_tokenizer_path))
target_tokenizer = Tokenizer.from_file(str(target_tokenizer_path))
print('test examples:', len(test))
print('columns:', test.column_names)
print('first prompt:', prompts[0])

In [ ]:
shared_tokenizer = Tokenizer.from_file(str(shared_tokenizer_path))
source_tokenizer = shared_tokenizer
target_tokenizer = shared_tokenizer

device = torch.device('cuda')
model = load_model(checkpoint_path, device)
parameters = sum(parameter.numel() for parameter in model.parameters())
print(f'shared vocab size: {shared_tokenizer.get_vocab_size()}')
print(f'parameters: {parameters / 1e6:.2f}M')
print('num_beams:', NUM_BEAMS)
print('max_new_tokens:', MAX_NEW_TOKENS)

In [ ]:
_ = generate_texts(
    model,
    source_tokenizer,
    target_tokenizer,
    prompts[:WARMUP_SIZE],
    batch_size=WARMUP_SIZE,
    max_source_length=model.config.max_source_length,
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=NUM_BEAMS,
    length_penalty=LENGTH_PENALTY,
)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

In [ ]:
predictions = []
start_time = time.perf_counter()
for start in tqdm(range(0, len(prompts), BATCH_SIZE)):
    predictions.extend(
        generate_texts(
            model,
            source_tokenizer,
            target_tokenizer,
            prompts[start:start + BATCH_SIZE],
            batch_size=BATCH_SIZE,
            max_source_length=model.config.max_source_length,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            length_penalty=LENGTH_PENALTY,
        )
    )
torch.cuda.synchronize()
elapsed = time.perf_counter() - start_time
result = {
    'gpu': torch.cuda.get_device_name(),
    'examples': len(predictions),
    'seconds': elapsed,
    'examples_per_second': len(predictions) / elapsed,
    'peak_memory_gb': torch.cuda.max_memory_allocated() / 1024**3,
    'fits_180_seconds': elapsed <= 180,
}
print(json.dumps(result, indent=2))
assert len(predictions) == len(test)

In [ ]:
output_path = Path('/kaggle/working/predictions.jsonl')
with output_path.open('w', encoding='utf-8') as file:
    for prediction in predictions:
        row = {'code': prediction}
        file.write(json.dumps(row, ensure_ascii=False) + '\n')
print('saved:', output_path)
print('first prediction:', predictions[0])

Замер включает токенизацию, генерацию и декодирование всех 1919 тестовых примеров, но не включает скачивание датасета и создание модели. Файл содержит ровно 1919 строк вида `{"code": "..."}` в исходном порядке. Необученная модель почти никогда не генерирует EOS, поэтому результат близок к worst-case. Если возникает CUDA OOM, уменьшите `BATCH_SIZE`.